# ML-HW10: RAG по резюме

Локальная RAG: FAISS + `qwen3:4b-instruct` (Ollama), 18 обезличенных резюме, две базы (с тегами / без), эксперименты с параметрами поиска и генерации.


## 1. Импорты


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TQDM_DISABLE'] = '1'

import pandas as pd
from IPython.display import display

import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import faiss
import httpx
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer

LAB_DIR = Path.cwd().resolve()
if not (LAB_DIR / 'data' / 'resumes_selected.csv').exists():
    cand = LAB_DIR / 'labs' / 'ML-HW10'
    if (cand / 'data' / 'resumes_selected.csv').exists():
        LAB_DIR = cand
ROOT = LAB_DIR.parents[1]
LAB_DATA_DIR = LAB_DIR / "data"
# отобранные 10–20 резюме рядом с лабой; fallback — полный датасет
SELECTED_CSV = LAB_DATA_DIR / "resumes_selected.csv"
SOURCE_CSV = ROOT / "data" / "resumes_ru_6000_labeled.csv"
DATA_CSV = SELECTED_CSV if SELECTED_CSV.exists() else SOURCE_CSV
# ASCII-путь: FAISS на Windows ломается на кириллице в пути
CACHE_DIR = Path(os.environ.get("RAG_CACHE_DIR", str(Path.home() / ".cache" / "mo3_ml_hw10_rag")))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = os.getenv("LLM_BASE_URL", os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")).rstrip("/")
OLLAMA_MODEL = os.getenv("LLM_MODEL", os.getenv("OLLAMA_MODEL", "qwen3:4b-instruct"))

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
NO_INFO_RU = (
    "В базе нет релевантной информации по этому запросу "
    "(нет такого кандидата / категории / факта)."
)
_NO_INFO_MARKERS = (
    "в базе нет релевантной",
    "нет релевантной информации",
    "нет информации по этому запросу",
    "информации нет",
)


@dataclass
class GenParams:
    temperature: float = 0.45
    top_k: int = 50
    top_p: float = 0.9
    max_new_tokens: int = 140


_tokenizer = None
_embed_model = None


def get_embedder():
    global _tokenizer, _embed_model
    if _tokenizer is None:
        # без jupyter-widgets прогресс-баров в выводе ноутбука
        os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
        os.environ.setdefault("TQDM_DISABLE", "1")
        try:
            from transformers.utils import logging as hf_logging

            hf_logging.disable_progress_bar()
        except Exception:
            pass
        _tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
        _embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME)
        _embed_model.eval()
    return _tokenizer, _embed_model


def embed_texts(texts: Iterable[str], batch_size: int = 32) -> np.ndarray:
    tokenizer, model = get_embedder()
    vectors = []
    batch: list[str] = []
    for text in texts:
        batch.append(text)
        if len(batch) >= batch_size:
            vectors.append(_embed_batch(batch, tokenizer, model))
            batch = []
    if batch:
        vectors.append(_embed_batch(batch, tokenizer, model))
    return np.vstack(vectors).astype("float32")


def _embed_batch(batch: list[str], tokenizer, model) -> np.ndarray:
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        emb = model(**inputs).last_hidden_state.mean(dim=1)
    return emb.cpu().numpy()


def build_faiss(embeddings: np.ndarray) -> faiss.IndexFlatL2:
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index


def is_relevant(distance: float, threshold: float = 40.0) -> bool:
    return float(distance) < threshold


def extract_candidate_ids(query: str) -> list[int]:
    """Достаёт номера кандидатов из вопроса: «кандидат 1», «№1», «кандидата №3»."""
    patterns = [
        r"кандидат(?:а|у|ом|е)?\s*№?\s*(\d+)",
        r"№\s*(\d+)",
        r"\bno\.?\s*(\d+)\b",
        r"\bid\s*[:=]?\s*(\d+)\b",
    ]
    ids: list[int] = []
    for pat in patterns:
        for m in re.finditer(pat, query, flags=re.IGNORECASE):
            cid = int(m.group(1))
            if cid not in ids:
                ids.append(cid)
    return ids


def find_docs_by_candidate_id(documents: list[str], candidate_id: int) -> list[str]:
    needle = f"Кандидат №{candidate_id}"
    return [doc for doc in documents if doc.startswith(needle + "\n") or doc.startswith(needle + " ")]


_SEARCH_STOP_PREFIX = re.compile(
    r"^(?:"
    r"сколько|какие|какой|какая|какое|каких|каким|какими|"
    r"кто|кого|кому|кем|"
    r"что|чем|чего|где|когда|как|"
    r"есть\s+ли|имеется\s+ли|"
    r"у\s+каких|у\s+кого|"
    r"перечисли|назови|найди|подбери"
    r")\b[\s,.:;−–—-]*",
    re.IGNORECASE,
)


def search_text(query: str) -> str:
    """Только для FAISS: предмет поиска без служебной вопросительной оболочки.
    На ответ это не влияет — в модель уходит полный исходный вопрос."""
    q = (query or "").strip()
    for _ in range(3):
        nxt = _SEARCH_STOP_PREFIX.sub("", q).strip()
        if nxt == q:
            break
        q = nxt
    q = q.strip(" ?!.…")
    return q or query.strip()


def retrieve(
    query: str,
    index: faiss.Index,
    documents: list[str],
    top_k: int = 3,
) -> tuple[list[str], np.ndarray]:
    """Явный № кандидата + семантика по тексту резюме. Категории не используем.

    В индекс идём по search_text(query), чтобы «какие …» и «сколько …»
    находили одни и те же резюме. Формулировка вопроса для ответа не меняется.
    """
    id_docs: list[str] = []
    for cid in extract_candidate_ids(query):
        id_docs.extend(find_docs_by_candidate_id(documents, cid))

    q = embed_texts([search_text(query)])
    distances, indices = index.search(q, max(top_k, 5))
    semantic_docs = [documents[i] for i in indices[0] if i >= 0]
    semantic_dists = [float(x) for x in distances[0]]

    docs: list[str] = []
    dists: list[float] = []

    def _add(doc: str, dist: float):
        if doc in docs:
            return
        docs.append(doc)
        dists.append(dist)

    for doc in id_docs:
        _add(doc, 0.0)
    for doc, dist in zip(semantic_docs, semantic_dists):
        _add(doc, dist)

    limit = max(top_k, len(id_docs), 5)
    return docs[:limit], np.asarray(dists[:limit], dtype="float32")


def ollama_generate(
    prompt: str,
    params: GenParams | None = None,
    system: str | None = None,
    model: str | None = None,
) -> str:
    params = params or GenParams()
    model = model or OLLAMA_MODEL
    system = system or (
        "Ты ассистент по базе резюме. Отвечай на вопрос только по тексту резюме в контексте. "
        "Игнорируй строку «Категория: ...», если она есть. "
        "Отвечай прямо по формулировке вопроса (число, список, да/нет — как спрашивают), "
        "плюс короткое почему по фактам. 1–2 предложения. "
        "Если точной роли нет, но опыт близкий — предложи таких кандидатов; "
        "отказ только если совсем мимо. Не выдумывай. Либо ответ, либо отказ — не оба."
    )
    payload = {
        "model": model,
        "stream": False,
        "think": False,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        "options": {
            "temperature": params.temperature,
            "top_k": params.top_k,
            "top_p": params.top_p,
            "num_predict": params.max_new_tokens,
            # иначе Ollama поднимает огромный ctx (65k) и ответ идёт минутами
            "num_ctx": int(os.getenv("OLLAMA_NUM_CTX", "4096")),
        },
    }
    base = os.getenv("LLM_BASE_URL", os.getenv("OLLAMA_BASE_URL", OLLAMA_URL)).rstrip("/")
    with httpx.Client(timeout=300.0) as client:
        r = client.post(f"{base}/api/chat", json=payload)
        r.raise_for_status()
        data = r.json()
    return (data.get("message") or {}).get("content", "").strip()


def _is_refusal(text: str) -> bool:
    low = text.lower()
    return any(m in low for m in _NO_INFO_MARKERS)


def normalize_answer(text: str) -> str:
    """Либо отказ, либо факт — без гибрида «нет информации, но вот кандидат…»."""
    text = (text or "").strip()
    if not text:
        return NO_INFO_RU
    if not _is_refusal(text):
        return text

    # вырезаем фразы отказа и смотрим, остался ли отдельный ответ
    rest = text
    rest = re.sub(
        r"в базе нет релевантной информации[^.!?\n]*[.!?]?",
        " ",
        rest,
        flags=re.IGNORECASE,
    )
    rest = re.sub(
        r"нет релевантной информации[^.!?\n]*[.!?]?",
        " ",
        rest,
        flags=re.IGNORECASE,
    )
    rest = re.sub(
        r"\(нет такого кандидата\s*/\s*категории\s*/\s*факта\)\.?",
        " ",
        rest,
        flags=re.IGNORECASE,
    )
    rest = re.sub(r"^\s*(однако|но|хотя|тем не менее|всё же|все же)[,:\s]+", "", rest, flags=re.I)
    rest = re.sub(r"\s+", " ", rest).strip(" \n\t-—;:,.")

    # остался содержательный ответ про кандидатов — берём только его
    if rest and re.search(r"кандидат|категори|опыт|резюме", rest, flags=re.I):
        return rest
    return NO_INFO_RU


def rag_answer(
    query: str,
    index: faiss.Index,
    documents: list[str],
    top_k: int = 3,
    distance_threshold: float = 40.0,
    params: GenParams | None = None,
    lang: str = "ru",
    model: str | None = None,
) -> dict:
    asked_ids = extract_candidate_ids(query)
    # Явно спросили про номер, которого нет в базе — сразу отказ, без LLM
    if asked_ids:
        missing = [cid for cid in asked_ids if not find_docs_by_candidate_id(documents, cid)]
        if missing and len(missing) == len(asked_ids):
            return {
                "answer": NO_INFO_RU,
                "retrieved": [],
                "relevant": [],
                "distances": [],
                "used_llm": False,
            }

    retrieved, distances = retrieve(query, index, documents, top_k=top_k)
    relevant = [
        doc for doc, dist in zip(retrieved, distances) if is_relevant(dist, distance_threshold)
    ]
    if not relevant:
        return {
            "answer": NO_INFO_RU,
            "retrieved": retrieved,
            "relevant": [],
            "distances": [float(x) for x in distances],
            "used_llm": False,
        }

    context = "\n\n".join(f"[{i+1}] {doc}" for i, doc in enumerate(relevant))
    id_hint = ""
    if asked_ids:
        id_hint = (
            f" В вопросе указан кандидат №{', №'.join(map(str, asked_ids))}. "
            "Отвечай ТОЛЬКО про этого кандидата, не подменяй номер."
        )
    prompt = (
        f"Контекст:\n{context}\n\n"
        f"Вопрос: {query}\n\n"
        "Ответь прямо на вопрос в 1–2 предложениях по тексту опыта "
        "(не обходи формулировку вопроса стороной). "
        "Если роли нет дословно — ближайшие по опыту."
        f"{id_hint}\n"
        "Формат — одно из двух:\n"
        "1) короткий ответ с обоснованием;\n"
        f"2) ровно: {NO_INFO_RU}\n"
        "Не совмещай отказ и ответ."
    )
    answer = normalize_answer(ollama_generate(prompt, params=params, model=model))
    return {
        "answer": answer,
        "retrieved": retrieved,
        "relevant": relevant,
        "distances": [float(x) for x in distances],
        "used_llm": True,
    }


def anonymize_text(text: str) -> str:
    text = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[email]", text)
    text = re.sub(r"(?:\+7|8)[\s\-]?\(?\d{3}\)?[\s\-]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}", "[phone]", text)
    return text.strip()


def sample_resumes(n: int = 18, seed: int = 42) -> pd.DataFrame:
    """По заданию 10–20 обезличенных резюме. Равномерно по категориям из полного CSV."""
    df = pd.read_csv(SOURCE_CSV)
    per_class = max(1, n // df["class"].nunique())
    parts = []
    for _, g in df.groupby("class", sort=True):
        parts.append(g.sample(n=min(per_class, len(g)), random_state=seed))
    out = pd.concat(parts, ignore_index=True)
    if len(out) < n:
        rest = df.loc[~df.index.isin(out.index)]
        need = n - len(out)
        if len(rest) > 0:
            out = pd.concat(
                [out, rest.sample(n=min(need, len(rest)), random_state=seed)],
                ignore_index=True,
            )
    out = out.head(n).reset_index(drop=True)
    out["candidate_id"] = out.index + 1
    return out


def load_selected_frame() -> pd.DataFrame:
    """Читает уже сохранённую выборку из labs/ML-HW10/data/."""
    if not SELECTED_CSV.exists():
        raise FileNotFoundError(
            f"Нет {SELECTED_CSV}. Сначала запустите prepare_data.py"
        )
    df = pd.read_csv(SELECTED_CSV)
    if "candidate_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["candidate_id"] = df.index + 1
    return df


def load_resume_documents(
    with_tags: bool = True,
    n: int = 18,
    seed: int = 42,
) -> tuple[list[str], pd.DataFrame]:
    if SELECTED_CSV.exists():
        df = load_selected_frame().head(n).reset_index(drop=True)
    else:
        df = sample_resumes(n=n, seed=seed)
    docs = []
    for _, row in df.iterrows():
        raw = anonymize_text(str(row["text"]).replace("\r", "\n"))
        cid = int(row["candidate_id"])
        label = str(row["class"])
        if with_tags:
            docs.append(f"Кандидат №{cid}\nКатегория: {label}\n{raw}")
        else:
            docs.append(f"Кандидат №{cid}\n{raw}")
    return docs, df


def _faiss_write(index: faiss.Index, dest: Path) -> None:
    """FAISS на Windows плохо пишет по Unicode-путям — через %TEMP%."""
    import tempfile

    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(tempfile.gettempdir()) / f"faiss_{os.getpid()}_{dest.name}"
    faiss.write_index(index, str(tmp))
    dest.write_bytes(tmp.read_bytes())
    tmp.unlink(missing_ok=True)


def _faiss_read(src: Path) -> faiss.Index:
    import tempfile

    tmp = Path(tempfile.gettempdir()) / f"faiss_read_{os.getpid()}_{src.name}"
    tmp.write_bytes(src.read_bytes())
    try:
        return faiss.read_index(str(tmp))
    finally:
        tmp.unlink(missing_ok=True)


def save_index(name: str, documents: list[str], embeddings: np.ndarray, index: faiss.Index) -> Path:
    folder = CACHE_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    np.save(folder / "embeddings.npy", embeddings)
    (folder / "documents.json").write_text(
        json.dumps(documents, ensure_ascii=False), encoding="utf-8"
    )
    _faiss_write(index, folder / "index.faiss")
    return folder


def load_index(name: str) -> tuple[list[str], faiss.Index] | None:
    folder = CACHE_DIR / name
    if not (folder / "index.faiss").exists():
        return None
    index = _faiss_read(folder / "index.faiss")
    documents = json.loads((folder / "documents.json").read_text(encoding="utf-8"))
    return documents, index


def ensure_resume_indices(force: bool = False, n: int = 18) -> dict:
    result = {}
    for name, with_tags in [("resumes_with_tags", True), ("resumes_no_tags", False)]:
        if not force:
            loaded = load_index(name)
            if loaded is not None and len(loaded[0]) == n:
                docs, index = loaded
                result[name] = {"documents": docs, "index": index, "n": n}
                continue
        docs, meta = load_resume_documents(with_tags=with_tags, n=n)
        emb = embed_texts(docs, batch_size=16)
        index = build_faiss(emb)
        save_index(name, docs, emb, index)
        meta_path = CACHE_DIR / name / "meta.csv"
        meta.to_csv(meta_path, index=False, encoding="utf-8-sig")
        result[name] = {"documents": docs, "index": index, "n": n, "meta": meta}
    return result

QUESTIONS = [
    {
        "q": "Кто из кандидатов умеет делать химическую завивку и окраску волос?",
        "kind": "has_answer",
        "expected": "Кандидат №1",
        "gold": "Кандидат №1 (в резюме: завивка, перманент, окраска волос).",
    },
    {
        "q": "У какого кандидата есть опыт полевых визитов к педиатрам, терапевтам и неврологам?",
        "kind": "has_answer",
        "expected": "Кандидат №3",
        "gold": "Кандидат №3 (медицинский промоушен / визиты к врачам).",
    },
    {
        "q": "Кто занимался продажей автомобильных запчастей и составлением коммерческих предложений?",
        "kind": "has_answer",
        "expected": "Кандидат №10",
        "gold": "Кандидат №10 (продажа автозапчастей, КП, договоры).",
    },
    {
        "q": "Есть ли в базе кандидат с опытом работы поваром?",
        "kind": "no_answer",
        "expected": "",
        "gold": "Повара в выборке нет — модель должна сказать, что информации нет.",
    },
]

HOMEWORK_QUESTIONS = [QUESTIONS[0], QUESTIONS[1], QUESTIONS[3]]

print('модель:', OLLAMA_MODEL)
print('данные:', DATA_CSV)


## 2. Данные

18 резюме (по 3 на категорию) в `data/resumes_selected.csv`. В индекс — сырой текст с меткой `Кандидат №N`.


In [2]:
print(LAB_DATA_DIR)
print(SELECTED_CSV.exists())

docs_tags, df = load_resume_documents(with_tags=True)
docs_plain, _ = load_resume_documents(with_tags=False)
display(df[['candidate_id', 'class']])
print(
    f'выборка: {len(df)} резюме, {df["class"].nunique()} категорий, '
    f'по {int(df.groupby("class").size().min())}–{int(df.groupby("class").size().max())} на класс.'
)
print('документов:', len(docs_tags))
print(docs_tags[0][:400])
print('---')
print(docs_plain[0][:400])


C:\Users\Zhenya\Desktop\МО 3\mo3_team_colabs\labs\ML-HW10\data
True


,candidate_id,class
0,1,Медицина и здравоохранение
1,2,Медицина и здравоохранение
2,3,Медицина и здравоохранение
3,4,Образование и наука
4,5,Образование и наука
5,6,Образование и наука
6,7,Строительство и ремонт
7,8,Строительство и ремонт
8,9,Строительство и ремонт
9,10,Торговля и продажи


выборка: 18 резюме, 6 категорий, по 3–3 на класс.
документов: 18
Кандидат №1
Категория: Медицина и здравоохранение
Расчесывание, стрижка волос взрослых и детей.

Укладка и завивка волос в соответствии с направлением моды и особенностями лица.

Завивка волос на бигуди, специальными зажимами и щипцами, химическим и электрическим способом (перманент).

Массаж и мытье головы.

Нанесение на волосы химических препаратов и растворов.

Окраска волос в различные цвета и
---
Кандидат №1
Расчесывание, стрижка волос взрослых и детей.

Укладка и завивка волос в соответствии с направлением моды и особенностями лица.

Завивка волос на бигуди, специальными зажимами и щипцами, химическим и электрическим способом (перманент).

Массаж и мытье головы.

Нанесение на волосы химических препаратов и растворов.

Окраска волос в различные цвета и оттенки, их обесцвечивание


## 3. FAISS: база с тегами и без тегов

In [3]:
packs = ensure_resume_indices(force=False)
for name, pack in packs.items():
    print(name, 'ntotal =', pack['index'].ntotal)
idx_tags = packs['resumes_with_tags']
idx_plain = packs['resumes_no_tags']


resumes_with_tags ntotal = 18
resumes_no_tags ntotal = 18


## 4. Проверочные вопросы

1. Кто умеет химическую завивку и окраску волос? → кандидат №1  
2. У кого полевые визиты к педиатрам, терапевтам и неврологам? → кандидат №3  
3. Есть ли кандидат с опытом работы поваром? → в базе нет


In [ ]:
QUESTIONS_HW = HOMEWORK_QUESTIONS
for item in QUESTIONS_HW:
    print(item['kind'], ':', item['q'])
    print('  эталон:', item['gold'])


## 5. Базовый RAG

In [5]:
BASE = GenParams(temperature=0.1, top_k=50, top_p=0.8, max_new_tokens=100)
DISTANCE_THRESHOLD = 55.0

for item in QUESTIONS_HW:
    out = rag_answer(
        item['q'],
        idx_tags['index'],
        idx_tags['documents'],
        top_k=3,
        distance_threshold=DISTANCE_THRESHOLD,
        params=BASE,
        lang='ru',
    )
    print('Q:', item['q'])
    print('distances:', out['distances'])
    print('relevant:', len(out['relevant']), '| used_llm:', out['used_llm'])
    print('A:', out['answer'])
    print('-' * 60)


W1107 23:06:18.833000 22772 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Q: Кто из кандидатов умеет делать химическую завивку и окраску волос?
distances: [9.457117080688477, 12.038254737854004, 12.626923561096191, 13.132616996765137, 13.142692565917969]
relevant: 5 | used_llm: True
A: Кандидат №1 умеет делать химическую завивку и окраску волос, так как в его опыте указано: «Завивка волос на бигуди, специальными зажимами и щипцами, химическим и электрическим способом (перманент). Окраска волос в различные цвета и оттенки, их обесцвечивание».
------------------------------------------------------------


Q: У какого кандидата есть опыт полевых визитов к педиатрам, терапевтам и неврологам?
distances: [13.440272331237793, 14.353874206542969, 14.36313247680664, 14.455164909362793, 14.555912017822266]
relevant: 5 | used_llm: True
A: 1) У кандидата №3 есть опыт полевых визитов к специалистам, включая педиатров, терапевтов и неврологов, поскольку в его обязанностях прямо указано: «Полевые визиты к специалистам» и перечислены целевые аудитории — педиатры, терапевты, неврологи.
------------------------------------------------------------


Q: Есть ли в базе кандидат с опытом работы поваром?
distances: [9.611769676208496, 10.034481048583984, 10.351226806640625, 10.388532638549805, 10.59630298614502]
relevant: 5 | used_llm: True
A: В базе нет релевантной информации по этому запросу (нет такого кандидата / категории / факта).
------------------------------------------------------------


## 6. Сравнение баз с тегами и без

In [6]:
rows = []
for name, pack in [('with_tags', idx_tags), ('no_tags', idx_plain)]:
    for item in QUESTIONS_HW:
        out = rag_answer(
            item['q'], pack['index'], pack['documents'],
            top_k=3, distance_threshold=DISTANCE_THRESHOLD, params=BASE, lang='ru',
        )
        rows.append({
            'base': name,
            'question': item['q'],
            'n_relevant': len(out['relevant']),
            'used_llm': out['used_llm'],
            'answer': out['answer'],
        })
cmp_df = pd.DataFrame(rows)
display(cmp_df)
same = (
    cmp_df.pivot(index='question', columns='base', values='answer')
    .assign(eq=lambda x: x['with_tags'] == x['no_tags'])['eq'].sum()
)
print(
    f'с тегами и без: совпало ответов {same}/{cmp_df["question"].nunique()}; '
    f'оба варианта находят №1/№3 и отказывают на поваре.'
)


,base,question,n_relevant,used_llm,answer
0,with_tags,Кто из кандидатов умеет делать химическую зави...,5,True,Кандидат №1 умеет делать химическую завивку и ...
1,with_tags,У какого кандидата есть опыт полевых визитов к...,5,True,1) У кандидата №3 есть опыт полевых визитов к ...
2,with_tags,Есть ли в базе кандидат с опытом работы поваром?,5,True,В базе нет релевантной информации по этому зап...
3,no_tags,Кто из кандидатов умеет делать химическую зави...,5,True,Кандидат №1 умеет делать химическую завивку и ...
4,no_tags,У какого кандидата есть опыт полевых визитов к...,5,True,1) У кандидата №3 есть опыт полевых визитов к ...
5,no_tags,Есть ли в базе кандидат с опытом работы поваром?,5,True,В базе нет релевантной информации по этому зап...


с тегами и без: совпало ответов 2/3; оба варианта находят №1/№3 и отказывают на поваре.


## 7. Эксперименты с параметрами

Один параметр за раз. Значения: `top_k_search ∈ {1,3,5,10}`, `temperature ∈ {0.1,0.4,0.7,1.0}` (3 повтора), `top_k_gen ∈ {5,10,50,100}`, `top_p ∈ {0.3,0.5,0.8,1.0}`.


In [ ]:
import time

import pandas as pd


OUT = CACHE_DIR / "experiments_resumes.csv"
N_RESUMES = 18


def evaluate_answer(kind: str, answer: str, expected: str) -> str:
    a = (answer or "").lower().replace("№", "№").replace("no", "№")
    a_norm = a.replace(" ", "")
    if kind == "no_answer":
        markers = [
            "нет релевантной",
            "нет информации",
            "в базе нет",
            "такой категории нет",
            "не найден",
            "отсутств",
            "нет такого",
            "не указан",
        ]
        return "ok: отказ без галлюцинации" if any(m in a for m in markers) else "плохо: возможная галлюцинация"
    if expected:
        exp = expected.lower().replace(" ", "")
        # «Кандидат №1» / «кандидат 1» / «№1»
        digits = "".join(ch for ch in expected if ch.isdigit())
        if exp in a_norm or (digits and (f"№{digits}" in a or f"кандидат№{digits}" in a_norm or f"кандидат {digits}" in a)):
            return "ok: есть ожидаемый факт"
        words = [w for w in expected.lower().replace("/", " ").split() if len(w) > 2]
        hits = sum(1 for w in words if w in a)
        if hits >= max(1, len(words) // 2):
            return "частично"
        return "слабо"
    return "проверено"


def run_suite(dataset: str, pack: dict, questions: list[dict], threshold: float) -> list[dict]:
    rows = []
    docs, index = pack["documents"], pack["index"]
    base = GenParams(temperature=0.1, top_k=50, top_p=0.8, max_new_tokens=100)

    def one(q, settings, top_k_search=3, params=None):
        params = params or base
        t0 = time.time()
        out = rag_answer(
            q["q"],
            index,
            docs,
            top_k=top_k_search,
            distance_threshold=threshold,
            params=params,
            lang="ru",
        )
        rows.append(
            {
                "dataset": dataset,
                "question": q["q"],
                "expected": q.get("expected", ""),
                "settings": settings,
                "answer": out["answer"],
                "distances": out["distances"],
                "n_relevant": len(out["relevant"]),
                "eval": evaluate_answer(q["kind"], out["answer"], q.get("expected", "")),
                "sec": round(time.time() - t0, 2),
                "model": OLLAMA_MODEL,
            }
        )
        print(f"  [{settings[:40]}] {q['q'][:50]}... -> {rows[-1]['eval']} ({rows[-1]['sec']}s)")

    for q in questions:
        one(q, "baseline top_k_search=3 temp=0.1 top_k_gen=50 top_p=0.8")

    for top_k in [1, 3, 5, 10]:
        for q in questions:
            one(q, f"top_k_search={top_k} (остальное baseline)", top_k_search=top_k)

    q0 = questions[0]
    for temp in [0.1, 0.4, 0.7, 1.0]:
        for rep in range(1, 4):
            one(
                q0,
                f"temperature={temp} repeat={rep}",
                params=GenParams(temperature=temp, top_k=50, top_p=0.8, max_new_tokens=100),
            )

    for top_k_gen in [5, 10, 50, 100]:
        for q in questions:
            one(
                q,
                f"top_k_gen={top_k_gen}",
                params=GenParams(temperature=0.1, top_k=top_k_gen, top_p=0.8, max_new_tokens=100),
            )

    for top_p in [0.3, 0.5, 0.8, 1.0]:
        for q in questions:
            one(
                q,
                f"top_p={top_p}",
                params=GenParams(temperature=0.1, top_k=50, top_p=top_p, max_new_tokens=100),
            )

    return rows


def main():
    print(f"Building resume FAISS indices (n={N_RESUMES})...")
    packs = ensure_resume_indices(force=True, n=N_RESUMES)
    for k, v in packs.items():
        print(k, len(v["documents"]))

    questions = HOMEWORK_QUESTIONS
    print("Questions:")
    for q in questions:
        print(" -", q["kind"], q["q"], "| gold:", q.get("gold", ""))

    rows = []
    print("Experiments on resumes_with_tags...")
    rows += run_suite("resumes_with_tags", packs["resumes_with_tags"], questions, threshold=55.0)

    print("Baseline comparison on resumes_no_tags...")
    docs = packs["resumes_no_tags"]["documents"]
    index = packs["resumes_no_tags"]["index"]
    base = GenParams(temperature=0.1, top_k=50, top_p=0.8, max_new_tokens=100)
    for q in questions:
        t0 = time.time()
        out = rag_answer(q["q"], index, docs, top_k=3, distance_threshold=55.0, params=base, lang="ru")
        rows.append(
            {
                "dataset": "resumes_no_tags",
                "question": q["q"],
                "expected": q.get("expected", ""),
                "settings": "baseline top_k_search=3 temp=0.1 top_k_gen=50 top_p=0.8",
                "answer": out["answer"],
                "distances": out["distances"],
                "n_relevant": len(out["relevant"]),
                "eval": evaluate_answer(q["kind"], out["answer"], q.get("expected", "")),
                "sec": round(time.time() - t0, 2),
                "model": OLLAMA_MODEL,
            }
        )
        print("  no_tags:", q["q"][:50], "->", rows[-1]["eval"])

    df = pd.DataFrame(rows)
    df.to_csv(OUT, index=False, encoding="utf-8-sig")
    print("Saved", OUT, "rows=", len(df))

exp_path = CACHE_DIR / 'experiments_resumes.csv'
if not exp_path.exists():
    main()
exp_df = pd.read_csv(exp_path)
exp_df['ok'] = exp_df['eval'].astype(str).str.startswith('ok')

def family(s):
    s = str(s)
    if s.startswith('baseline'):
        return 'baseline'
    if 'top_k_search' in s:
        return 'top_k_search'
    if 'temperature' in s:
        return 'temperature'
    if 'top_k_gen' in s:
        return 'top_k_gen'
    if 'top_p' in s:
        return 'top_p'
    return 'other'

exp_df['family'] = exp_df['settings'].map(family)
print('строк:', len(exp_df))
display(exp_df['family'].value_counts().to_frame('count'))
print(
    f'сводка прогонов: всего {len(exp_df)}, ok={exp_df["ok"].mean():.0%}, '
    f'семьи = {", ".join(exp_df["family"].value_counts().index)}.'
)
display(exp_df[['dataset', 'question', 'settings', 'n_relevant', 'eval', 'sec', 'answer']])
print(
    f'полная таблица экспериментов: время {exp_df.sec.min():.2f}–{exp_df.sec.max():.2f}s, '
    f'все eval начинаются с ok={exp_df["ok"].all()}.'
)


### 7.1. Baseline

In [8]:
base_view = exp_df[exp_df['family'] == 'baseline'][
    ['dataset', 'question', 'n_relevant', 'eval', 'sec', 'answer']
].reset_index(drop=True)
display(base_view)
ok_rate = base_view['eval'].astype(str).str.startswith('ok').mean()
print('ok_rate:', round(ok_rate, 3))
print(
    f'baseline: {len(base_view)} строк (2 базы × 3 вопроса), ok_rate={ok_rate:.0%}; '
    f'№1/№3 находятся, повар — отказ.'
)


,dataset,question,n_relevant,eval,sec,answer
0,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,1.19,Кандидат №1 умеет делать химическую завивку и ...
1,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,1.70,У Кандидата №3 есть опыт полевых визитов к пед...
2,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,1.51,В базе нет релевантной информации.
3,resumes_no_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,1.81,Кандидат №1 умеет делать химическую завивку и ...
4,resumes_no_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,2.26,У Кандидата №3 есть опыт полевых визитов к пед...
5,resumes_no_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,1.98,В базе нет релевантной информации.


ok_rate: 1.0
baseline: 6 строк (2 базы × 3 вопроса), ok_rate=100%; №1/№3 находятся, повар — отказ.


### 7.2. top_k в поиске

In [9]:
tk = exp_df[exp_df['family'] == 'top_k_search'].copy()
tk['top_k_search'] = tk['settings'].str.extract(r'top_k_search=(\d+)').astype(int)
display(tk.sort_values(['top_k_search', 'question'])[
    ['top_k_search', 'question', 'n_relevant', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('по каждому k три вопроса: смотрим, не ломается ли ответ при узком/широком контексте.')
display(tk.groupby('top_k_search')['ok'].mean().to_frame('ok_rate'))
print(f'ok_rate по k: везде {tk.groupby("top_k_search")["ok"].mean().min():.0%}–{tk.groupby("top_k_search")["ok"].mean().max():.0%}.')
display(tk.groupby('top_k_search')[['n_relevant', 'sec']].mean())
print(
    f'среднее: n_relevant растёт с k ({tk.groupby("top_k_search")["n_relevant"].mean().min():.0f}→'
    f'{tk.groupby("top_k_search")["n_relevant"].mean().max():.0f}), '
    f'время {tk.groupby("top_k_search")["sec"].mean().min():.2f}→'
    f'{tk.groupby("top_k_search")["sec"].mean().max():.2f}s.'
)


,top_k_search,question,n_relevant,eval,sec,answer
0,1,Есть ли в базе кандидат с опытом работы поваром?,1,ok: отказ без галлюцинации,0.94,В базе нет релевантной информации.
1,1,Кто из кандидатов умеет делать химическую зави...,1,ok: есть ожидаемый факт,1.39,Кандидат №1 умеет делать химическую завивку и ...
2,1,У какого кандидата есть опыт полевых визитов к...,1,ok: есть ожидаемый факт,1.61,У кандидата №3 есть опыт полевых визитов к пед...
3,3,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,0.87,В базе нет релевантной информации.
4,3,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,1.26,Кандидат №1 умеет делать химическую завивку и ...
5,3,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,2.42,У Кандидата №3 есть опыт полевых визитов к пед...
6,5,Есть ли в базе кандидат с опытом работы поваром?,5,ok: отказ без галлюцинации,1.67,В базе нет релевантной информации.
7,5,Кто из кандидатов умеет делать химическую зави...,5,ok: есть ожидаемый факт,1.67,Кандидат №1 умеет делать химическую завивку и ...
8,5,У какого кандидата есть опыт полевых визитов к...,5,ok: есть ожидаемый факт,2.11,У Кандидата №3 есть опыт полевых визитов к пед...
9,10,Есть ли в базе кандидат с опытом работы поваром?,10,ok: отказ без галлюцинации,2.76,В базе нет релевантной информации.


по каждому k три вопроса: смотрим, не ломается ли ответ при узком/широком контексте.


,ok_rate
top_k_search,
1,1.0
3,1.0
5,1.0
10,1.0


ok_rate по k: везде 100%–100%.


,n_relevant,sec
top_k_search,,
1,1.0,1.313333
3,3.0,1.516667
5,5.0,1.816667
10,10.0,3.446667


среднее: n_relevant растёт с k (1→10), время 1.31→3.45s.


### 7.3. temperature

In [10]:
tp = exp_df[exp_df['family'] == 'temperature'].copy()
tp['temperature'] = tp['settings'].str.extract(r'temperature=([0-9.]+)').astype(float)
tp['repeat'] = tp['settings'].str.extract(r'repeat=(\d+)').astype(int)
display(tp.sort_values(['temperature', 'repeat'])[
    ['temperature', 'repeat', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('один вопрос (завивка), по 3 повтора на temperature — насколько плывёт формулировка.')
for t, g in tp.groupby('temperature'):
    uniq = g['answer'].drop_duplicates().tolist()
    print(f'temp={t}: {len(uniq)} уникальных из {len(g)}')
    for a in uniq:
        print(' ', a)
nuniq = tp.groupby('temperature')['answer'].nunique()
print(
    f'итог: уникальных формулировок на температуру {int(nuniq.min())}–{int(nuniq.max())} '
    f'из 3 повторов.'
)


,temperature,repeat,eval,sec,answer
0,0.1,1,ok: есть ожидаемый факт,1.28,Кандидат №1 умеет делать химическую завивку и ...
1,0.1,2,ok: есть ожидаемый факт,1.16,Кандидат №1 умеет делать химическую завивку и ...
2,0.1,3,ok: есть ожидаемый факт,1.22,Кандидат №1 умеет делать химическую завивку и ...
3,0.4,1,ok: есть ожидаемый факт,1.20,Кандидат №1 умеет делать химическую завивку и ...
4,0.4,2,ok: есть ожидаемый факт,1.32,Кандидат №1 умеет делать химическую завивку и ...
5,0.4,3,ok: есть ожидаемый факт,1.32,Кандидат №1 умеет делать химическую завивку и ...
6,0.7,1,ok: есть ожидаемый факт,1.29,Кандидат №1 умеет делать химическую завивку и ...
7,0.7,2,ok: есть ожидаемый факт,1.32,Кандидат №1 умеет делать химическую завивку и ...
8,0.7,3,ok: есть ожидаемый факт,1.35,Кандидат №1 умеет делать химическую завивку и ...
9,1.0,1,ok: есть ожидаемый факт,1.29,Кандидат №1 умеет делать химическую завивку и ...


один вопрос (завивка), по 3 повтора на temperature — насколько плывёт формулировка.
temp=0.1: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
temp=0.4: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
temp=0.7: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
temp=1.0: 1 уникальных из 3
  Кандидат №1 умеет делать химическую завивку и окраску волос.
итог: уникальных формулировок на температуру 1–1 из 3 повторов.


### 7.4. top_k в генерации

In [11]:
tg = exp_df[exp_df['family'] == 'top_k_gen'].copy()
tg['top_k_gen'] = tg['settings'].str.extract(r'top_k_gen=(\d+)').astype(int)
display(tg.sort_values(['top_k_gen', 'question'])[
    ['top_k_gen', 'question', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('меняли только top_k генерации; факты (№1/№3/отказ) должны держаться.')
display(tg.groupby('top_k_gen')['ok'].mean().to_frame('ok_rate'))
print(f'ok_rate по top_k_gen: min={tg.groupby("top_k_gen")["ok"].mean().min():.0%}.')
display(tg.groupby('top_k_gen')['answer'].nunique().to_frame('unique_answers'))
print(
    f'уникальных ответов на значение: {int(tg.groupby("top_k_gen")["answer"].nunique().min())}–'
    f'{int(tg.groupby("top_k_gen")["answer"].nunique().max())} (обычно по числу вопросов).'
)


,top_k_gen,question,eval,sec,answer
0,5,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,1.45,В базе нет релевантной информации.
1,5,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.28,Кандидат №1 умеет делать химическую завивку и ...
2,5,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,2.03,У Кандидата №3 есть опыт полевых визитов к пед...
3,10,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.92,В базе нет релевантной информации.
4,10,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.29,Кандидат №1 умеет делать химическую завивку и ...
5,10,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.82,У Кандидата №3 есть опыт полевых визитов к пед...
6,50,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.95,В базе нет релевантной информации.
7,50,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.43,Кандидат №1 умеет делать химическую завивку и ...
8,50,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.96,У Кандидата №3 есть опыт полевых визитов к пед...
9,100,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.92,В базе нет релевантной информации.


меняли только top_k генерации; факты (№1/№3/отказ) должны держаться.


,ok_rate
top_k_gen,
5,1.0
10,1.0
50,1.0
100,1.0


ok_rate по top_k_gen: min=100%.


,unique_answers
top_k_gen,
5,3
10,3
50,3
100,3


уникальных ответов на значение: 3–3 (обычно по числу вопросов).


### 7.5. top_p

In [12]:
pp = exp_df[exp_df['family'] == 'top_p'].copy()
pp['top_p'] = pp['settings'].str.extract(r'top_p=([0-9.]+)').astype(float)
display(pp.sort_values(['top_p', 'question'])[
    ['top_p', 'question', 'eval', 'sec', 'answer']
].reset_index(drop=True))
print('то же для top_p: влияет ли nucleus sampling на ok и текст.')
display(pp.groupby('top_p')['ok'].mean().to_frame('ok_rate'))
print(f'ok_rate по top_p: min={pp.groupby("top_p")["ok"].mean().min():.0%}.')
display(pp.groupby('top_p')['answer'].nunique().to_frame('unique_answers'))
print(
    f'уникальных ответов на top_p: {int(pp.groupby("top_p")["answer"].nunique().min())}–'
    f'{int(pp.groupby("top_p")["answer"].nunique().max())}.'
)


,top_p,question,eval,sec,answer
0,0.3,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.95,В базе нет релевантной информации.
1,0.3,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.34,Кандидат №1 умеет делать химическую завивку и ...
2,0.3,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.89,У Кандидата №3 есть опыт полевых визитов к пед...
3,0.5,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.93,В базе нет релевантной информации.
4,0.5,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.38,Кандидат №1 умеет делать химическую завивку и ...
5,0.5,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.91,У Кандидата №3 есть опыт полевых визитов к пед...
6,0.8,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.86,В базе нет релевантной информации.
7,0.8,Кто из кандидатов умеет делать химическую зави...,ok: есть ожидаемый факт,1.25,Кандидат №1 умеет делать химическую завивку и ...
8,0.8,У какого кандидата есть опыт полевых визитов к...,ok: есть ожидаемый факт,1.72,У Кандидата №3 есть опыт полевых визитов к пед...
9,1.0,Есть ли в базе кандидат с опытом работы поваром?,ok: отказ без галлюцинации,0.84,В базе нет релевантной информации.


то же для top_p: влияет ли nucleus sampling на ok и текст.


,ok_rate
top_p,
0.3,1.0
0.5,1.0
0.8,1.0
1.0,1.0


ok_rate по top_p: min=100%.


,unique_answers
top_p,
0.3,3
0.5,3
0.8,3
1.0,3


уникальных ответов на top_p: 3–3.


### 7.6. С тегами / без тегов

In [13]:
tag_cmp = exp_df[exp_df['family'] == 'baseline'][
    ['dataset', 'question', 'n_relevant', 'eval', 'answer']
].reset_index(drop=True)
display(tag_cmp)
w = tag_cmp[tag_cmp.dataset == 'resumes_with_tags'].set_index('question')['answer']
n = tag_cmp[tag_cmp.dataset == 'resumes_no_tags'].set_index('question')['answer']
same = int((w == n).sum())
ok_t = tag_cmp.loc[tag_cmp.dataset == 'resumes_with_tags', 'eval'].astype(str).str.startswith('ok').mean()
ok_n = tag_cmp.loc[tag_cmp.dataset == 'resumes_no_tags', 'eval'].astype(str).str.startswith('ok').mean()
print(f'теги vs без: совпало формулировок {same}/{len(w)}, ok с тегами={ok_t:.0%}, без={ok_n:.0%}.')


,dataset,question,n_relevant,eval,answer
0,resumes_with_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,Кандидат №1 умеет делать химическую завивку и ...
1,resumes_with_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,У Кандидата №3 есть опыт полевых визитов к пед...
2,resumes_with_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,В базе нет релевантной информации.
3,resumes_no_tags,Кто из кандидатов умеет делать химическую зави...,3,ok: есть ожидаемый факт,Кандидат №1 умеет делать химическую завивку и ...
4,resumes_no_tags,У какого кандидата есть опыт полевых визитов к...,3,ok: есть ожидаемый факт,У Кандидата №3 есть опыт полевых визитов к пед...
5,resumes_no_tags,Есть ли в базе кандидат с опытом работы поваром?,3,ok: отказ без галлюцинации,В базе нет релевантной информации.


теги vs без: совпало формулировок 3/3, ok с тегами=100%, без=100%.


## 8. Демонстрация в web

In [14]:
print('Чат: http://127.0.0.1:8000')
print('RAG_BASE=resumes_no_tags, модель=', OLLAMA_MODEL)


Чат: http://127.0.0.1:8000
RAG_BASE=resumes_no_tags, модель= qwen3:4b-instruct


## 9. Выводы

Цифры ниже — из таблиц в разделах 6–7 (54 прогона, модель `qwen3:4b-instruct`, ok = 100%).

По baseline RAG находит нужных людей: завивка/окраска → кандидат №1, визиты к врачам → кандидат №3. На вопрос про повара отвечает отказом («в базе нет релевантной информации») и на базе с тегами, и без — повар не выдумывается. Это же видно по всем 14 прогонам с этим вопросом.

Сравнение `top_k` поиска (1 / 3 / 5 / 10): ok везде 100%, число релевантных документов растёт как k, время с ~1.3s до ~3.5s. Для коротких фактологических вопросов хватает k=3; больший k только раздувает контекст.

Temperature 0.1 / 0.4 / 0.7 / 1.0 (по 3 повтора на одном вопросе): во всех 12 ответах одна и та же формулировка про кандидата №1. На этих вопросах температура почти не меняет текст.

`top_k_gen` (5…100) и `top_p` (0.3…1.0) тоже держат ok=100%. Параметры генерации здесь вторичны относительно retrieval и порога релевантности.

База с тегами vs без тегов: на всех трёх baseline-вопросах ответы совпали, ok 100% в обоих случаях. Для этих фактов теги не обязательны — нужное уже есть в тексте резюме.

Для демо оставляю `top_k_search=3`, `top_k_gen=50`, `top_p≈0.8–0.9`, база `resumes_no_tags` (без категорий в индексе). Время ответов в таблице примерно 0.8–4.3s. Чат: http://127.0.0.1:8000.
